# 01 · Setup e tools do assistente de receitas

**Objetivo deste notebook:** montar as peças do agente e *ver* o perfil de custo/latência de cada uma.

O agente tem só **4 tools**, escolhidas para cobrir os três perfis de custo que aparecem em produção:

| Tool | Tipo | Custo | Latência | O que demonstra |
|---|---|---|---|---|
| `check_geladeira` | Determinística | US$ 0 | ~ms | dados dinâmicos, controle do cenário |
| `verificar_validade` | Determinística | US$ 0 | ~ms | nem toda etapa precisa de LLM |
| `buscar_receitas_similares` | RAG | embedding (centavos de centavo) + tokens dos docs | ~0,5 s | trade-off de `k`, grounding |
| `criar_receita` | LLM + Pydantic | depende do modelo | segundos | saída estruturada e validada |

O código fica no pacote `receitas/` (na raiz do projeto) para que os 5 notebooks usem **exatamente** as mesmas tools e avaliadores.

## 1. Setup

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # adiciona a raiz do projeto ao path

from receitas import config

## 2. Geladeira virtual procedural

`gerar_geladeira()` sorteia ingredientes de um catálogo de ~100 itens (10 categorias), com quantidade e validade.

- **Sem `seed`** → geladeira nova a cada execução (simula o mundo real, dinâmico).
- **Com `seed`** → geladeira idêntica (reprodutível — é o que usaremos no golden dataset).
- `n_itens` vai de poucos itens até o catálogo inteiro.

In [12]:
import pandas as pd
from receitas.geladeira import gerar_geladeira, TOTAL_INGREDIENTES_CATALOGO

print("Ingredientes no catálogo:", TOTAL_INGREDIENTES_CATALOGO)

a, b = gerar_geladeira(15), gerar_geladeira(15)
print("Sem seed -> seeds sorteadas:", a["seed"], "e", b["seed"], "| iguais?", a == b)
print("Com seed=7 duas vezes -> iguais?", gerar_geladeira(15, seed=7) == gerar_geladeira(15, seed=7))
print("Pedindo 100 itens ->", len(gerar_geladeira(100)["itens"]), "itens")

geladeira = gerar_geladeira(n_itens=25)  # geladeira usada no notebook
pd.DataFrame(geladeira["itens"]).head(10)

Ingredientes no catálogo: 102
Sem seed -> seeds sorteadas: 318155 e 546449 | iguais? False
Com seed=7 duas vezes -> iguais? True
Pedindo 100 itens -> 100 itens


,nome,quantidade,unidade,categoria,validade
0,água de coco,400.0,ml,bebidas e outros,2026-11-27
1,bife de alcatra,400.0,g,carnes,2026-10-08
2,carne moída,950.0,g,carnes,2026-10-08
3,linguiça toscana,550.0,g,carnes,2026-10-06
4,presunto,550.0,g,carnes,2026-10-01
5,cebolinha,1.0,maço,ervas frescas,2026-10-07
6,manjericão,2.0,maço,ervas frescas,2026-10-04
7,banana,11.0,un,frutas,2026-10-01
8,maçã,1.0,un,frutas,2026-10-05
9,morango,800.0,g,frutas,2026-10-07


## 3. Tools determinísticas (custo zero, milissegundos)

A `geladeira` **não é argumento que o LLM preenche**: ela vem do estado do grafo via `InjectedState`.
Assim o modelo não consegue "inventar" a geladeira, e o schema que ele vê fica menor (menos tokens).
Para testar a tool isolada, passamos o estado na mão.

Note também a decisão de design em `verificar_validade`: itens **vencidos** são para **descartar** (segurança alimentar);
só os que **vão vencer** são prioridade.

In [13]:
import time
from receitas.tools import check_geladeira, verificar_validade

print("O que o LLM enxerga de check_geladeira:", check_geladeira.tool_call_schema.model_json_schema()["properties"])

def medir_ms(tool, args, repeticoes=200):
    inicio = time.perf_counter()
    for _ in range(repeticoes):
        tool.func(**args)  # chama a função direto, sem callbacks
    return (time.perf_counter() - inicio) / repeticoes * 1000

print(f"check_geladeira:    {medir_ms(check_geladeira, {'geladeira': geladeira}):.3f} ms por chamada")
print(f"verificar_validade: {medir_ms(verificar_validade, {'geladeira': geladeira}):.3f} ms por chamada\n")
print(verificar_validade.invoke({"geladeira": geladeira, "dias_limite": 3}))

O que o LLM enxerga de check_geladeira: {'categoria': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Categoria'}}
check_geladeira:    0.022 ms por chamada
verificar_validade: 0.061 ms por chamada

Hoje é 2026-10-02.
VENCIDOS (descartar, NÃO usar por segurança alimentar):
- presunto: 550 g (validade 2026-10-01)
- banana: 11 un (validade 2026-10-01)
- couve-flor: 11 un (validade 2026-10-01)
- repolho: 3 un (validade 2026-10-01)
- tomate: 7 un (validade 2026-10-01)
VENCENDO em até 3 dias (usar com prioridade):
- batata: 7 un (validade 2026-10-02) -> vence em 0 dia(s)
- manteiga: 250 g (validade 2026-10-03) -> vence em 1 dia(s)
- massa fresca: 700 g (validade 2026-10-03) -> vence em 1 dia(s)
- manjericão: 2 maço (validade 2026-10-04) -> vence em 2 dia(s)
- maçã: 1 un (validade 2026-10-05) -> vence em 3 dia(s)


## 4. RAG: base real de receitas + Chroma local

**Fonte:** [`Hieu-Pham/kaggle_food_recipes`](https://huggingface.co/datasets/Hieu-Pham/kaggle_food_recipes) — ~13,5 mil receitas reais do Epicurious (licença CC-BY-SA 3.0), em inglês.
Indexamos uma amostra fixa de 2.000 (configurável em `config.N_RECEITAS_INDEXADAS`).

- O que vira embedding: **título + ingredientes** (é o que importa para "achar receitas parecidas com o que tenho").
- As instruções ficam nos metadados, para o LLM ler quando a receita for recuperada.
- O índice é salvo em `data/chroma/`: **o custo de indexação é pago uma única vez**.
- A busca é cross-lingual: perguntamos em português/inglês e os embeddings aproximam os significados.

In [14]:
from receitas import rag

receitas_df = rag.carregar_receitas()
print("Receitas carregadas:", len(receitas_df))
display(receitas_df.head(3))

estimativa = rag.estimar_custo_indexacao(receitas_df)
print(f"Estimativa ANTES de indexar: {estimativa['tokens']:,} tokens -> US$ {estimativa['custo_usd']:.4f}")

Receitas carregadas: 2000


,titulo,ingredientes,instrucoes
0,Creamy Jalapeño Sauce,5-6 jalapeños (depending on heat and personal ...,"Pulse jalapeños, garlic, lime juice, and salt ..."
1,Salted Plum Lychee Panna Cotta,1 cup water; 2 teaspoons salted plum powder; 6...,Because agar-agar is seaweed-based and fairly ...
2,Fried Chicken Thighs with Cheesy Grits,1 1/2 cups buttermilk; 1 teaspoon cayenne pepp...,"Combine buttermilk, cayenne, garlic powder, sa..."


Estimativa ANTES de indexar: 250,895 tokens -> US$ 0.0050


In [15]:
# Cria o índice (só na primeira execução) e registra o custo no LangSmith
rag.construir_indice(receitas_df)

{'status': 'índice já existia, nada a pagar', 'documentos': 2000}

### O trade-off de `k`

`k` = quantas receitas recuperar. O embedding da consulta custa o mesmo para qualquer `k`;
o que cresce é o **texto devolvido ao LLM** (tokens de entrada), e isso é cobrado a cada chamada seguinte do agente.

In [16]:
from receitas.config import MODELO_FRONTEIRA, MODELO_SIMPLES, custo_usd
from receitas.tools import buscar_receitas_similares

linhas = []
for k in [1, 3, 5, 8]:
    inicio = time.perf_counter()
    texto, uso = buscar_receitas_similares.func("mushroom risotto with parmesan", k=k)
    tokens_contexto = rag.contar_tokens(texto)
    linhas.append({
        "k": k,
        "latência busca (s)": round(time.perf_counter() - inicio, 3),
        "custo embedding (US$)": uso["custo_embedding_usd"],
        "tokens enviados ao LLM": tokens_contexto,
        f"custo extra no {MODELO_FRONTEIRA} (US$)": round(custo_usd(MODELO_FRONTEIRA, tokens_contexto), 5),
        f"custo extra no {MODELO_SIMPLES} (US$)": round(custo_usd(MODELO_SIMPLES, tokens_contexto), 6),
    })
display(pd.DataFrame(linhas))
print(texto[:600])

,k,latência busca (s),custo embedding (US$),tokens enviados ao LLM,custo extra no gpt-4.1 (US$),custo extra no gpt-4.1-nano (US$)
0,1,1.376,1.800000e-07,204,0.00041,0.000020
1,3,0.554,1.800000e-07,732,0.00146,0.000073
2,5,0.606,1.800000e-07,1203,0.00241,0.000120
3,8,0.295,1.800000e-07,1979,0.00396,0.000198


### Turkey and Mushroom Risotto
Ingredients: 8 cups Day-After Turkey Stock; 4 tablespoons (1/2 stick) unsalted butter, divided; 1 small yellow onion, thinly sliced; 2 cups assorted fresh mushrooms (preferably wild, but white buttons will do), thinly sliced; 2 cups arborio or carnaroli rice; 1 cup shredded leftover turkey meat (optional); 1/2 cup shredded Parmesan; Kosher salt and freshly ground black pepper; 2 tablespoons finely chopped flat-leaf parsley
Preparo (resumo): Bring stock to a simmer in a medium pot over medium heat. Reduce heat to low. Cover and keep warm.
Meanwhile, melt 3 tables


**Leitura:** o embedding é praticamente grátis; o custo real do RAG é o contexto que ele injeta no prompt.
Por isso o agente usa `k=3` por padrão e o prompt manda chamar a busca **só quando o usuário pede inspiração** —
para um "o que faço com o que tenho?", RAG é custo sem ganho de qualidade.

## 5. `criar_receita`: LLM + saída estruturada validada por Pydantic

`with_structured_output(Receita)` força o modelo a responder no schema e o **Pydantic valida** tipos e limites
(ex.: `tempo_preparo_min > 0`, pelo menos 2 passos). É o contrato que um frontend consumiria.

A tool usa `response_format="content_and_artifact"`:
- **content** (curto) volta para o agente → poucos tokens;
- **artifact** (a receita completa, validada) fica disponível para o app, sem passar pelo LLM de novo.

In [17]:
from receitas.tools import criar_receita
from receitas.schemas import Receita
from receitas.geladeira import classificar_validade

validade = classificar_validade(geladeira)
disponiveis = [i["nome"] for i in geladeira["itens"] if i["nome"] not in {v["nome"] for v in validade["vencidos"]}]
prioritarios = [i["nome"] for i in validade["vencendo"]]

inicio = time.perf_counter()
mensagem = criar_receita.invoke({
    "type": "tool_call", "id": "teste", "name": "criar_receita",  # formato de tool call para receber o artifact
    "args": {"modelo": config.MODELO_INTERMEDIARIO, "pedido": "jantar rápido",
             "ingredientes": disponiveis, "ingredientes_prioritarios": prioritarios},
})
print(f"Latência: {time.perf_counter() - inicio:.2f} s\n")
print("O que o AGENTE recebe (content):\n", mensagem.content, "\n")
receita = Receita.model_validate(mensagem.artifact["receita"])
print("O que o FRONTEND recebe (artifact, validado):")
print(receita.model_dump_json(indent=2))

Latência: 2.72 s

O que o AGENTE recebe (content):
 Receita criada e validada: 'Massa fresca ao molho de manteiga com manjericão e batatas douradas' (30 min, 2 porções). Prioritários usados: batata, manteiga, massa fresca, manjericão. O app já exibe a receita completa ao usuário. 

O que o FRONTEND recebe (artifact, validado):
{
  "nome": "Massa fresca ao molho de manteiga com manjericão e batatas douradas",
  "descricao": "Um jantar rápido e saboroso com massa fresca ao molho de manteiga e manjericão, acompanhada de batatas douradas crocantes.",
  "ingredientes": [
    "200 g de massa fresca",
    "2 batatas médias",
    "2 colheres de sopa de manteiga",
    "1 punhado de folhas de manjericão",
    "Sal a gosto",
    "Pimenta-do-reino a gosto",
    "1 colher de sopa de azeite"
  ],
  "modo_preparo": [
    "Descasque as batatas e corte em cubos pequenos.",
    "Em uma frigideira, aqueça o azeite e doure as batatas até ficarem crocantes e macias. Tempere com sal e pimenta.",
    "Cozinh

E se o LLM devolver algo fora do contrato? O Pydantic barra **antes** de chegar ao usuário:

In [18]:
from pydantic import ValidationError

receita_quebrada = {**receita.model_dump(), "tempo_preparo_min": 0, "modo_preparo": ["só um passo"]}
try:
    Receita.model_validate(receita_quebrada)
except ValidationError as erro:
    print(erro)

2 validation errors for Receita
modo_preparo
  List should have at least 2 items after validation, not 1 [type=too_short, input_value=['só um passo'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short
tempo_preparo_min
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


## 6. O agente completo

`criar_alvo()` monta o grafo e devolve uma função `alvo(inputs) -> outputs` — o mesmo formato que o
`evaluate()` do LangSmith usa nos próximos notebooks. Os outputs já trazem **custo (US$), tokens e latência**
daquela execução, medidos com `get_usage_metadata_callback` (soma todos os LLMs chamados, inclusive o de dentro da tool).

In [20]:
from receitas.agente import criar_alvo
import json

alvo = criar_alvo(modelo=config.MODELO_FRONTEIRA)
saida = alvo({
    "pergunta": "Quero um jantar que aproveite o que está vencendo, mas pode completar com outras coisas da geladeira.",
    "geladeira": geladeira,
})

print("Tools chamadas:", saida["tools_chamadas"])
print(f"Custo: US$ {saida['custo_usd']:.5f} | tokens in/out: {saida['tokens_entrada']}/{saida['tokens_saida']} | latência: {saida['latencia_s']} s")
print("\nResposta final:", saida["resposta"])
print("\nReceita (contrato do frontend):")
print(json.dumps(saida["receita"], ensure_ascii=False, indent=2))

Tools chamadas: ['verificar_validade', 'check_geladeira', 'criar_receita']
Custo: US$ 0.01685 | tokens in/out: 5942/621 | latência: 7.87 s

Resposta final: Sugiro "Massa Fresca ao Molho de Batata, Manjericão e Manteiga", que aproveita batata, manteiga, massa fresca e manjericão, todos perto do vencimento. Assim, você evita desperdício e ainda faz um jantar saboroso. Outros ingredientes da geladeira complementam o prato. Receita completa já disponível no app!

Receita (contrato do frontend):
{
  "nome": "Massa Fresca ao Molho de Batata, Manjericão e Manteiga",
  "descricao": "Um prato cremoso e aromático que aproveita batata, manteiga, massa fresca e manjericão, perfeito para um jantar reconfortante.",
  "ingredientes": [
    "250 g de massa fresca",
    "2 batatas médias descascadas e picadas",
    "2 colheres de sopa de manteiga",
    "1/2 xícara de folhas de manjericão fresco",
    "1 dente de alho picado",
    "Sal a gosto",
    "Pimenta-do-reino a gosto",
    "Água para cozinhar",


## 7. Onde ver isso no LangSmith

Abra [smith.langchain.com](https://smith.langchain.com) → **Projects** → `assistente-receitas` e clique no trace `assistente_receitas`:

- a árvore mostra cada nó do grafo e cada tool com **latência própria**;
- runs de LLM mostram **tokens e custo**; o run `embedding_da_consulta` mostra o custo do embedding que registramos via `usage_metadata`;
- o run `busca_vetorial_chroma` mostra os documentos recuperados (grounding auditável);
- as tools determinísticas aparecem com latência de milissegundos e custo zero — o contraste fica visível na própria árvore.

In [21]:
# Aguarda o envio dos traces
from langchain_core.tracers.langchain import wait_for_all_tracers
wait_for_all_tracers()